# Exploração Trabalho 
Aqui vamos explorar os dados rellacionados a postos de trabalho criados e ocupação de pessoas imigrantes no Brasil. Dados fornecidos pelo CAGED obtidos em (https://www.gov.br/mj/pt-br/assuntos/seus-direitos/migracoes/portal-de-imigracao-laboral/obmigra-1/microdados/caged/CAGED)

In [ ]:
# Carregando uma amostra do CAGED 2025

import pandas as pd
from pathlib import Path

pasta = Path("../data/raw/trabalho")

for arquivo in pasta.rglob("*.csv"):
    print(arquivo.name)

def agregar_movimentacoes(dados, chaves):
    """Resume admissões, desligamentos e saldo pelas chaves informadas."""
    return dados.groupby(chaves).agg(
        admissoes=("saldomovimentacao", lambda valores: (valores == 1).sum()),
        desligamentos=("saldomovimentacao", lambda valores: (valores == -1).sum()),
        saldo=("saldomovimentacao", "sum"),
    )

In [ ]:
# Lendo as primeiras linhas do CAGED 2025

df_2025 = pd.read_csv(
    pasta / "RAIS_CTPS_CAGED_2025_MOV.csv",
    sep = ";",
    encoding = "utf-8",
    low_memory = False
)

display(df_2025.head(5))

In [ ]:
# Verificando a coluna "saldomovimentacao" pra inconsistências

print(df_2025["saldomovimentacao"].value_counts(dropna=False))

In [ ]:
# Verificando a distribuição mensal de movimentações

df_2025["competenciamov"].value_counts().sort_index()

### Verificar os códigos dos municípios
No SISMIGRA, utilizamos códigos IBGE de 7 dígitos. Já no CAGED, o exemplo 110002 possui apenas 6 dígitos.

In [ ]:
print(df_2025["municipio"].astype(str).str.len().value_counts())

print("\nExemplos de códigos:")
print(df_2025["municipio"].unique()[:15])

Não devemos simplesmente acrescentar um número ao código. Vamos criar uma tabela de correspondência usando os códigos oficiais do IBGE que já baixamos.

In [ ]:
# Carregando a tabela municipios_ibge criada no notebook 01 e salva em /processed

ibge_municipios = pd.read_parquet(
    "../data/processed/municipios_ibge.parquet"
)

# Criando código de 6 dígitos utilizado pelo CAGED 

ibge_municipios["codigo_caged"] = (
    ibge_municipios["codigo_ibge"] // 10

)

display(ibge_municipios.head())

# Verificando se há códigos duplicados 

print(
    "Códigos CAGED duplicados:",
    ibge_municipios["codigo_caged"].duplicated().sum()
)

In [ ]:
# Associando o CAGED 2025 aos códigos IBGE

df_2025 = df_2025.merge(
    ibge_municipios[["codigo_caged", "codigo_ibge"]],
    left_on="municipio",
    right_on="codigo_caged",
    how="left",
    validate="many_to_one"
)

print("Total de registros:", len(df_2025))
print("Com código IBGE:", df_2025["codigo_ibge"].notna().sum())
print("Sem código IBGE:", df_2025["codigo_ibge"].isna().sum())

In [ ]:
# Identificando os códigos não encontrados

df_2025[df_2025["codigo_ibge"].isna()]["municipio"].value_counts().head(20)

In [ ]:
# Excluindo da análise municipal os registros onde "municipio" == 999999

df_2025 = df_2025.dropna(
    subset=["codigo_ibge"]
).copy()

print("Registros restantes:", len(df_2025))

### Calcular admissões, demissões e saldo por município

In [ ]:
emprego_2025 = agregar_movimentacoes(df_2025, 'codigo_ibge').reset_index()

display(emprego_2025.head(10))

In [ ]:
# Adicionando os nomes dos municípios e UFs

emprego_2025 = emprego_2025.merge(
    ibge_municipios[["codigo_ibge", "municipio", "uf"]],
    on="codigo_ibge",
    how="left"
)

emprego_2025 = emprego_2025[["codigo_ibge", "municipio", "uf", "admissoes", "desligamentos", "saldo"]]

display(
    emprego_2025.sort_values("admissoes", ascending=False).head(20)
)

In [ ]:
# Validando os totais

print("Registros originais: ", len(df_2025))

print("Admissões + desligamentos:", emprego_2025["admissoes"].sum() + emprego_2025["desligamentos"].sum())

print("Saldo total:", emprego_2025["saldo"].sum())

### Analisando os dados de 2026

In [ ]:
# Carregando o DataFrame de 2026

df_2026 = pd.read_csv(
    pasta / "RAIS_CTPS_CAGED_2026_MOV.csv",
    sep = ";",
    encoding = "utf-8",
    low_memory=False
)

# Analisando os meses de 2026

print("Total de registros:", len(df_2026))
print("\nMovimentações por mês:")
print(df_2026["competenciamov"].value_counts().sort_index())

In [ ]:
# Comparando os períodos entre 2025 e 2026

# Selecionando janeiro a julho de 2025

caged_2025_jul = df_2025[
    df_2025["competenciamov"].between(202501, 202507)
]

for ano, df in [(2025, caged_2025_jul), (2026, df_2026)]:
    print(f"\n{ano}")
    print("Admissões:", (df["saldomovimentacao"] == 1).sum())
    print("Desligamentos:", (df["saldomovimentacao"] == -1).sum())
    print("Saldo:", df["saldomovimentacao"].sum())

In [ ]:
# Padronizando os municípios de 2026

df_2026 = df_2026.merge(
    ibge_municipios[["codigo_caged", "codigo_ibge"]],
    left_on="municipio",
    right_on="codigo_caged", 
    how="left",
    validate="many_to_one"
)

print("Total de resigstos:", len(df_2026))
print("Com código IBGE:", df_2026["codigo_ibge"].notna().sum())
print("Sem código IBGE:", df_2026["codigo_ibge"].isna().sum())

In [ ]:
#Verificando os registros nos dados de 2026 onde não há código IBGE

df_2026[df_2026["codigo_ibge"].isna()]["municipio"].value_counts().head(20)

In [ ]:
# Excluindo da análise municipal os registros onde "municipio" == 999999

df_2026 = df_2026.dropna(
    subset=["codigo_ibge"]
).copy()

print("Registros restantes:", len(df_2026))

In [ ]:
# Comparando janeiro a julho dos dois anos, considerando apenas movimentaç~es com municipio identificado

for ano, df in [(2025, caged_2025_jul), (2026, df_2026)]:
    admissoes = (df["saldomovimentacao"] == 1).sum()
    desligamentos = (df["saldomovimentacao"] == -1).sum()

    print(f"\n{ano} (jan-jul)")
    print("Admissões:", admissoes)
    print("Desligamentos:", desligamentos)
    print("Saldo:", admissoes - desligamentos)

In [ ]:
# Calculando indicadores municipais de 2026

emprego_2026 = agregar_movimentacoes(df_2026, 'codigo_ibge').reset_index()

display(emprego_2026.head(10))

In [ ]:
# Adicionando os nomes dos municípios e UFs à tabela de 2026

emprego_2026 = emprego_2026.merge(
    ibge_municipios[["codigo_ibge", "municipio", "uf"]],
    on="codigo_ibge",
    how="left"
)

emprego_2026 = emprego_2026[["codigo_ibge", "municipio", "uf", "admissoes", "desligamentos", "saldo"]]

display(
    emprego_2026.sort_values("admissoes", ascending=False).head(20)
)

In [ ]:
# Criando uma tabela equivalente para janeiro a julho de 2025, comparando municípios no mesmo período

emprego_2025_jul = agregar_movimentacoes(caged_2025_jul, 'codigo_ibge').reset_index()

display(emprego_2025_jul.head())

In [ ]:
# Juntando os indicadores municipais entre 2025 e 2026, usando o código do IBGE

comparacao_emprego = emprego_2025_jul.merge(
    emprego_2026,
    on="codigo_ibge",
    how="outer",
    suffixes=("_2025", "_2026")
)

display(comparacao_emprego.head(10))

In [ ]:
# Tratando valores ausentes 

colunas = [
    "admissoes_2025", "desligamentos_2025", "saldo_2025",
    "admissoes_2026", "desligamentos_2026", "saldo_2026",
]

comparacao_emprego[colunas] = comparacao_emprego[colunas].fillna(0)

display(comparacao_emprego.head(10))

In [ ]:
# Completando os nomes dos municípios e UFs que estavam NaN, utilizando a tabela do IBGE

nomes = ibge_municipios.set_index("codigo_ibge")

comparacao_emprego["municipio"] = (
    comparacao_emprego["municipio"].fillna(
        comparacao_emprego["codigo_ibge"].map(nomes["municipio"])
    )
)

comparacao_emprego["uf"] = (
    comparacao_emprego["uf"].fillna(
        comparacao_emprego["codigo_ibge"].map(nomes["uf"])
    )
)

print(comparacao_emprego[["municipio", "uf"]].isna().sum())

Calculando a variação das admissões por município entre janeiro e julho de 2025 e 2026

In [ ]:
comparacao_emprego["variacao_absoluta"] = (
    comparacao_emprego["admissoes_2026"] - comparacao_emprego["admissoes_2025"]
)

comparacao_emprego["variacao_%"] = (
    comparacao_emprego["variacao_absoluta"] / comparacao_emprego["admissoes_2025"].replace(0, float("nan")) * 100
)

display(
    comparacao_emprego[["uf", "municipio", "admissoes_2025", "admissoes_2026", "variacao_absoluta", "variacao_%"]].sort_values(
        "variacao_absoluta", ascending=False
    ).head(20)
)

In [ ]:
# Validando os totais

for ano, df in[(2025, caged_2025_jul), (2026, df_2026)]:
    original = (df["saldomovimentacao"] == 1).sum()
    consolidado = comparacao_emprego[f"admissoes_{ano}"].sum()

    print(f"{ano}:")
    print("Original:", original)
    print("Consolidado:", int(consolidado))
    print("Diferença:", int(original - consolidado), "\n")

In [ ]:
# Verificando desligamentos e saldos

for ano, df in [(2025, caged_2025_jul), (2026, df_2026)]:
    print(f"\n{ano}")

    for indicador in ["desligamentos", "saldo"]:
        if indicador == "desligamentos":
            original = (df["saldomovimentacao"] == -1).sum()
        else:
            original = df["saldomovimentacao"].sum()

        consolidado = comparacao_emprego[f"{indicador}_{ano}"].sum()

        print(indicador, "— Diferença:", int(original - consolidado))

### Salvando os indicadores de emprego

In [ ]:
pasta_destino = Path("../data/processed/trabalho")
pasta_destino.mkdir(parents=True, exist_ok=True)

comparacao_emprego.to_parquet(
    pasta_destino / "indicadores_emprego_2025_2026.parquet",
    index=False
)

print("Indicadores de emprego salvos!")

# O que oferecer? 
Análise das ocupações que mais contratam trabalhadores migrantes

In [ ]:
# Identificando as 20 ocupações com mais admissões em 2025

admissoes_2025 = df_2025[
    df_2025["saldomovimentacao"] == 1
]

display(
    admissoes_2025["cbo2002ocupacao"].value_counts().head(20)
)

Temos as ocupações que mais contrataram, mas precisamos da descrição dessas ocupações. Para isso, vamos carregar a tabela de correspondências "codigo_ocupacao_cbo". 

In [ ]:
# Carregando a tabela com código e descrição das ocupações

ocupacoes_cbo = pd.read_csv(
    "../data/raw/trabalho/cbo/cbo2002-ocupacao.csv",
    sep=";",
    encoding="latin-1",
    low_memory=False
)

In [ ]:
# Inspeção inicial da tabela

print("Número de linhas:", ocupacoes_cbo.shape[0])
print("Número de colunas:", ocupacoes_cbo.shape[1])
print("Colunas:", ocupacoes_cbo.columns.tolist())
print("Tipo de dados da coluna Código:", ocupacoes_cbo["CODIGO"].dtype)
display(ocupacoes_cbo.head(5))

In [ ]:
# Verificando o tipo de dados em df_2025["cbo2002ocupacao"] e se há duplicatas em ocupacoes_cbo["Código"]

print(df_2025["cbo2002ocupacao"].dtype)
print("Códigos duplicados em ocupacoes_cbo[Codigo]:", ocupacoes_cbo["CODIGO"].duplicated().sum())

In [ ]:
# Fazendo o merge para incorporar a descrição das ocupações à tabela df_2025, criando uma nova tabela

caged_ocupacoes = df_2025.merge(
    ocupacoes_cbo,
    left_on="cbo2002ocupacao",
    right_on="CODIGO",
    how="left",
    validate="many_to_one"
)

print(caged_ocupacoes["TITULO"].isna().sum())

In [ ]:
# Verificando quais códigos não possuem correspondência de ocupação

caged_ocupacoes[caged_ocupacoes["TITULO"].isna()]["cbo2002ocupacao"].value_counts()

In [ ]:
caged_ocupacoes[caged_ocupacoes["TITULO"].isna()]["cbo2002ocupacao"].nunique()

In [ ]:
# Tratando os valores ausentes substituindo-os por "Não identificado"

caged_ocupacoes["TITULO"] = caged_ocupacoes["TITULO"].fillna("Não identificado")

print(caged_ocupacoes["TITULO"].isna().sum())

Ranqueando as ocupações

In [ ]:
ranking_ocupacoes_2025 = caged_ocupacoes[
    caged_ocupacoes["saldomovimentacao"] == 1
].groupby("TITULO").size().sort_values(ascending=False).reset_index(name="admissoes")

display(ranking_ocupacoes_2025.head(20))

Admissões, desligamentos e saldo por ocupação

In [ ]:
# Exibindo as 20 ocupações com maior saldo em 2025 (admissões - desligamentos)

emprego_por_ocupacao_2025 = agregar_movimentacoes(caged_ocupacoes, 'TITULO')

display(emprego_por_ocupacao_2025.sort_values(by="saldo", ascending=False).head(20))

In [ ]:
# Validando os resultados

print("Total admissões:", emprego_por_ocupacao_2025["admissoes"].sum())
print("Total desligamentos:", emprego_por_ocupacao_2025["desligamentos"].sum())
print("Saldo total:", emprego_por_ocupacao_2025["saldo"].sum())

In [ ]:
# Identificando ocupações com mais desligamentos

display(emprego_por_ocupacao_2025.sort_values(by="desligamentos", ascending=False).head(10))

In [ ]:
# Identificando ocupações com saldo negativo

display(emprego_por_ocupacao_2025.sort_values(by="saldo", ascending=True).head(10))

Importante: saldo negativo não significa necessariamente que uma profissão esteja em declínio. Pode refletir sazonalidade ou rotatividade de trabalhadores.

### Analisando os setores econômicos (CNAE)

In [ ]:
caged_ocupacoes[
    caged_ocupacoes["saldomovimentacao"] == 1
].groupby("secao").size().sort_values(ascending=False).head(10)

## Atrelando os códigos CNAE as descrições

In [ ]:
# Carregando a tabela de códigos CNAE (disponível em https://cnae.ibge.gov.br/classificacoes/download-concla/8265-download)

secoes_cnae = pd.read_csv(
    "../data/raw/trabalho/cnae/cnae_secoes.csv"
)

print("Shape:", secoes_cnae.shape)
print("Colunas:", secoes_cnae.columns)
display("5 primeiras linhas:", secoes_cnae.head())

In [ ]:
# Verificando se as colunas secao das duas tabelas são compatíveis

print("Datatype de caged_ocupacoes[secao]:", caged_ocupacoes["secao"].dtype)
print("Datatype de secoes_cnae[secao]:", secoes_cnae["secao"].dtype)
print("\nLinhas duplicadas:", secoes_cnae["secao"].duplicated().sum())

In [ ]:
# Associando as das tabelas através do .merge

caged_completo = caged_ocupacoes.merge(
    secoes_cnae,
    left_on="secao",
    right_on="secao",
    how="left",
    validate="many_to_one"
)

print("Número de linhas:", len(caged_completo))
print("Valores ausentes:", caged_completo["descricao_secao"].isna().sum())

In [ ]:
emprego_por_setor_2025 = agregar_movimentacoes(caged_completo, 'descricao_secao')

display(emprego_por_setor_2025.sort_values("saldo",ascending=False).head())

In [ ]:
# Validando os indicadores

print("Total admissoes:", emprego_por_setor_2025["admissoes"].sum())
print("Total desligamentos:", emprego_por_setor_2025["desligamentos"].sum())
print("Total saldo:", emprego_por_setor_2025["saldo"].sum())

Analisando as subclasses CNAE

In [ ]:
print(caged_completo["subclasse"].dtype)

In [ ]:
print("Entrdas únicas:", caged_completo["subclasse"].nunique())
print("\n5 subclasses mais comuns:", caged_completo["subclasse"].value_counts().head(5))

In [ ]:
# Carregando a tabela com descrição das subclasses

subclasses_cnae = pd.read_csv("../data/raw/trabalho/cnae/cnae_subclasses.csv",
                              sep=";",
                              encoding="utf-8",
                              low_memory=False)

# Verificando o número de linhas, colunas e códigos únicos

print("Número de linhas:", subclasses_cnae.shape[0])
print("Número de colunas:", subclasses_cnae.shape[1])
print("Número de códigos únicos:", subclasses_cnae["codigo_numerico"].nunique())

In [ ]:
# Preparando para o cruzamento com CAGED

print("Tipo de dados 'codigo_numerico':", subclasses_cnae["codigo_numerico"].dtype)
print("Tipo de dados 'subclasse':", caged_completo["subclasse"].dtype)
print(
    "Valores ausentes em 'codigo_numerico':",
    subclasses_cnae["codigo_numerico"].isna().sum()
)

In [ ]:
# Fazendo o merge 

caged_atividades = caged_completo.merge(
    subclasses_cnae,
    left_on="subclasse",
    right_on="codigo_numerico",
    how="left",
    validate="many_to_one"
)

print("Tamanho de caged_atividades:", len(caged_atividades))
print("Subclasses sem correspondencia:", caged_atividades["descricao"].isna().sum())

In [ ]:
# Investigando os 122 registros sem correspondência

caged_atividades[
    caged_atividades["descricao"].isna()
]["subclasse_x"].value_counts()

8630505 foi eliminado da CNAE 2.0 por uma resolução da CONCLA em 2007. Isso explica sua ausência na tabela atual. 

9900800 está relacionado à seção U — Organismos internacionais e outras instituições extraterritoriais. Ainda precisamos verificar sua correspondência exata na versão utilizada.

In [ ]:
# Verificando quandos CNAE diferentes aparecem em caged_atividades

print("Número de grupos diferentes:", caged_atividades["grupo"].nunique())

In [ ]:
# Verificando valores ausentes na coluna "grupo"

print("Valores ausentes em grupo:" , caged_atividades["grupo"].isna().sum())

### Calculando indicadores por grupo CNAE

In [ ]:
# Criando uma tabela que agrega dados de emprego por grupo CNAE

emprego_por_grupo_2025 = agregar_movimentacoes(caged_atividades, 'grupo')

In [ ]:
# Verificando os 10 grupos com maior saldo (admissoes - desligamentos)

emprego_por_grupo_2025.sort_values("saldo", ascending=False).head(10)

In [ ]:
# Preparando para fazer o merge

print("Valores úncos em subclasses_cnae['grupo']:", subclasses_cnae["grupo"].nunique())
print("Número de linhas após aplicar .dropduplicates:", len(subclasses_cnae.drop_duplicates(subset="grupo")))

In [ ]:
# Preparando a tabela de referencia 

# Carregando a tabela cnae_grupos.csv

grupos_cnae = pd.read_csv(
    "../data/raw/trabalho/cnae/cnae_grupos.csv",
    sep=";",
    encoding="utf-8",
    low_memory=False
)

In [ ]:
# Exploração inicial da tabela grupos_cnae

display(grupos_cnae.head())
print(grupos_cnae.dtypes)
print("Dimensões:", grupos_cnae.shape)

In [ ]:
# Verificando a unicidade da chave

print("Grupos únicos:", grupos_cnae["grupo"].nunique())
print("Grupos duplicados:", grupos_cnae["grupo"].duplicated().sum())

In [ ]:
# Adicionando as descricoes ao ranking

ranking_grupos_2025 = emprego_por_grupo_2025.reset_index(names="grupo").merge(
    grupos_cnae,
    left_on="grupo",
    right_on="grupo",
    how="left",
    validate="many_to_one"
)

display(ranking_grupos_2025.sort_values(by="saldo", ascending=False))

In [ ]:
# Validando o cruzamento

print("Número de lnhas em ranking_grupos_2025:", len(ranking_grupos_2025))
print("Descrições ausentes:", ranking_grupos_2025["descricao"].isna().sum())

### Salvando os resultados

In [ ]:
ranking_grupos_2025 = ranking_grupos_2025.sort_values(by="saldo", ascending=False)

In [ ]:
ranking_grupos_2025.to_parquet(path="../data/processed/trabalho/emprego_por_grupo_2025.parquet")

### Emprego por município e grupo CNAE
Quais atividades mais contratam trabalhadores migrantes em cada município brasileiro? 

In [ ]:
# Verificando as colunas necessárias 

caged_atividades[
    ["codigo_ibge", "grupo", "saldomovimentacao"]
].head(10)

In [ ]:
# Verificando valores ausentes

caged_atividades[
    ["codigo_ibge", "grupo", "saldomovimentacao"]
].isna().sum()

In [ ]:
emprego_por_municipio_grupo_2025 = agregar_movimentacoes(caged_atividades, ['codigo_ibge', 'grupo']).reset_index()

display(emprego_por_municipio_grupo_2025.sort_values(by="saldo",ascending=False).head(10))

In [ ]:
# Validando os resultados

print("Admissões:", emprego_por_municipio_grupo_2025["admissoes"].sum())
print("Desligamentos:", emprego_por_municipio_grupo_2025["desligamentos"].sum())
print("Saldo:", emprego_por_municipio_grupo_2025["saldo"].sum())

print("Combinações município-grupo:", len(emprego_por_municipio_grupo_2025))
print("Municípios distintos:", emprego_por_municipio_grupo_2025["codigo_ibge"].nunique())

Adicionando os nomes dos municípios

In [ ]:
# Padroniza a chave antes da integração territorial.
emprego_por_municipio_grupo_2025["codigo_ibge"] = (
    emprego_por_municipio_grupo_2025["codigo_ibge"].astype(int)
)

In [ ]:
# Fazendo o merge para adicionar municípios e UFs

emprego_por_municipio_grupo_2025 = emprego_por_municipio_grupo_2025.merge(
    ibge_municipios,
    left_on="codigo_ibge",
    right_on="codigo_ibge",
    how="left",
    validate="many_to_one"
)

display(emprego_por_municipio_grupo_2025)
print(emprego_por_municipio_grupo_2025.columns)

In [ ]:
# Limpando as colunas duplicadas 
emprego_por_municipio_grupo_2025 = emprego_por_municipio_grupo_2025[["codigo_ibge", "municipio", "uf", "grupo", "admissoes", "desligamentos", "saldo"]]

print(emprego_por_municipio_grupo_2025.columns)
display(emprego_por_municipio_grupo_2025.head())

In [ ]:
# Adicionando descrições dos grupos CNAE

emprego_por_municipio_grupo_2025 = emprego_por_municipio_grupo_2025.merge(
    grupos_cnae[["grupo", "descricao"]],
    on="grupo",
    how="left",
    validate="many_to_one"
)

display(emprego_por_municipio_grupo_2025)

In [ ]:
# Validando o merge

print("Colunas:", emprego_por_municipio_grupo_2025.columns)

print("Número de linhas,colunas:", emprego_por_municipio_grupo_2025.shape)

print(
    "Descrições ausentes:",
    emprego_por_municipio_grupo_2025["descricao"].isna().sum()
)

display(emprego_por_municipio_grupo_2025.head())

In [ ]:
# Validação final dos indicadores

print("Total de admissoes:", emprego_por_municipio_grupo_2025["admissoes"].sum())
print("Total de desligamentos:", emprego_por_municipio_grupo_2025["desligamentos"].sum())
print("Saldo total:", emprego_por_municipio_grupo_2025["saldo"].sum())

Nossa tabela emprego_por_municipio_grupo_2025 está pronta, com 29.601 combinações de município e grupo CNAE, devidamente identificadas e descritas

In [ ]:
# Salvado em Parquet

emprego_por_municipio_grupo_2025.to_parquet(
    "../data/processed/trabalho/emprego_por_municipio_grupo_2025.parquet",
    index=False
)

### Integrar emprego e migração

Agora vamos começar a conectar as duas principais dimensões do projeto UNIESF:
- SISMIGRA: registros de migrantes por município.
- CAGED: oportunidades de emprego por município e atividade econômica.

In [ ]:
# Carregando a tabela de indicadores migratórios 

indicadores_migracao = pd.read_parquet(
    "../data/processed/migracao/indicadores_migracao_municipal.parquet"
)

display(indicadores_migracao.head())
print("\nColunas:", indicadores_migracao.columns)
print("\nNúmero de linhas, colunas:", indicadores_migracao.shape)

In [ ]:
# Verificando a chave de integração (codigo_ibge)

print(indicadores_migracao["codigo_ibge"].dtype)
print(indicadores_migracao["codigo_ibge"].duplicated().sum())
print(indicadores_migracao["codigo_ibge"].isna().sum())

In [ ]:
# Verificando a cobertura geográfica (quantos municípios aparecem nas duas bases).

indicadores_migracao["codigo_ibge"].isin(emprego_por_municipio_grupo_2025["codigo_ibge"]).sum()

Encontramos 760 municípios presentes tanto no SISMIGRA quanto no CAGED 2025.
O que isso significa para o projeto?
Temos uma decisão metodológica importante: não devemos perder automaticamente os municípios que aparecem em apenas uma das bases.

Um município com registros migratórios, mas sem movimentações de emprego identificadas, ainda pode representar uma oportunidade para a UNIESF.

Por isso, minha recomendação é considerar um outer merge na construção da base municipal integrada, preservando os 1.228 municípios.

In [ ]:
emprego_por_municipio_grupo_2025.head()

In [ ]:
# Preparando o CAGED para integração

emprego_municipal_2025 = emprego_por_municipio_grupo_2025.groupby(by='codigo_ibge').agg(
    admissoes=('admissoes', 'sum'),
    desligamentos=('desligamentos', 'sum'),
    saldo=('saldo', 'sum')
).reset_index()

print('Número de linhas:', len(emprego_municipal_2025))
print('Duplicados:', emprego_municipal_2025['codigo_ibge'].duplicated().sum())
print(emprego_municipal_2025[['admissoes', 'desligamentos', 'saldo']].sum())
display(emprego_municipal_2025.head())

### Integrando SISMIGRA e CAGED

In [ ]:
# Fazendo o merge

indicadores_integrados = indicadores_migracao.merge(
    emprego_municipal_2025,
    on='codigo_ibge',
    how='outer',
    validate='one_to_one',
    indicator=True
)

In [ ]:
indicadores_integrados["_merge"].value_counts()

In [ ]:
# Investigando valores ausentes 

indicadores_integrados.isna().sum()

Um ponto importante: variacao_% possui 187 valores ausentes além dos 294 municípios exclusivos do CAGED.

In [ ]:
# Investigando 'variacao_%'

indicadores_integrados[
    (indicadores_integrados['_merge'] != 'right_only') & (indicadores_integrados['variacao_%'].isna() & (indicadores_integrados['2024'] == 0))
][['municipio', 'uf', '2024', '2025', 'variacao_%']].shape

Confirmamos a hipótese: os 187 municípios com variacao_% ausente possuem zero registros em 2024.

Isso significa que não encontramos outros motivos para os valores ausentes nessa coluna.

Nossa decisão metodológica será manter esses valores como NaN, pois a variação percentual não pode ser calculada com o denominador igual a zero.

### Completar os nomes dos municípios
Agora vamos resolver os 294 municípios exclusivos do CAGED que ficaram sem municipio e uf após o outer merge.

In [ ]:
# Quantos dos 1228 municípios presentes em indicadores_integrados estão presentes em ibge_municipios

indicadores_integrados['codigo_ibge'].isin(ibge_municipios['codigo_ibge']).sum()

In [ ]:
indicadores_integrados = indicadores_integrados.merge(
    ibge_municipios[['codigo_ibge','municipio', 'uf']],
    on='codigo_ibge',
    how='left', 
    validate='many_to_one',
    suffixes=('', '_ibge')
)

In [ ]:
indicadores_integrados['municipio'] = indicadores_integrados['municipio'].fillna(indicadores_integrados['municipio_ibge'])
indicadores_integrados['uf'] = indicadores_integrados['uf'].fillna(indicadores_integrados['uf_ibge'])

print(indicadores_integrados.isna().sum())

In [ ]:
# Removendo colunas auxiliares

indicadores_integrados.drop(columns={'municipio_ibge', 'uf_ibge'}, inplace=True)

In [ ]:
print(indicadores_integrados.columns)
print(indicadores_integrados.shape)

### Interpretando os indicadores ausentes
Antes de decidir como tratar os NaN, precisamos distinguir duas situações:
- Ausência de registro na fonte: um município aparece no SISMIGRA, mas não no CAGED, ou vice-versa.
- Valor zero: a fonte registra explicitamente zero para determinado indicador.
Essa distinção é importante para não interpretar ausência de dados como ausência de oportunidades.


In [ ]:
# Investigando municípios exclusivos do SISMIGRA

indicadores_integrados[
    indicadores_integrados['_merge'] == 'left_only'
][['municipio', 'uf', '2025', 'admissoes', 'desligamentos', 'saldo']].head(10)

In [ ]:
# Investigando municípios exclusivos do CAGED

indicadores_integrados[
    indicadores_integrados['_merge'] == 'right_only'
][['municipio', 'uf', '2025', 'admissoes', 'desligamentos', 'saldo']].head(10)

In [ ]:
# Validando a consistência dos indicadores

# Validando a consistência dos indicadores de emprego

emprego_validacao = indicadores_integrados[
    indicadores_integrados["admissoes"].notna()
]

divergencias = (
    emprego_validacao["admissoes"] - emprego_validacao["desligamentos"]
    != emprego_validacao["saldo"]
)

print("Municípios verificados:", len(emprego_validacao))
print("Municípios com divergências:", divergencias.sum())

In [ ]:
# Salvando a base integrada

indicadores_integrados["_merge"] = indicadores_integrados["_merge"].astype(str)

indicadores_integrados.to_parquet(
    "../data/processed/indicadores_integrados_municipais.parquet",
    index=False
)

## Análise exploratória integrada

In [ ]:
# Municipios com mais registros migratórios em 2025

indicadores_integrados.sort_values('2025', ascending=False).head(10)

In [ ]:
# Identificando os 10 municípios com maior saldo de empregos formais para imigrantes em 2025

indicadores_integrados[['municipio', 'uf', '2025', 'admissoes', 'desligamentos', 'saldo']].sort_values('saldo', ascending=False).head(10)

Principais observações
- Curitiba lidera a geração líquida de empregos formais para migrantes (+7.422), superando São Paulo (+6.489).
- Paraná e Santa Catarina concentram 7 dos 10 municípios com maiores saldos.
- Chapecó, Cascavel e Joinville apresentam saldos expressivos, apesar de terem menos registros migratórios que grandes capitais.
- Boa Vista, segunda colocada em registros migratórios, não aparece entre os dez maiores saldos de emprego.
Isso reforça que precisamos considerar migração e mercado de trabalho como dimensões distintas na priorização territorial.

In [ ]:
# Investigando municípios com saldo negativo

indicadores_integrados[['municipio', 'uf', '2025', 'admissoes', 'desligamentos', 'saldo']].sort_values('saldo', ascending=True).head(10)

In [ ]:
indicadores_integrados[['municipio', 'uf', '2025', 'admissoes', 'desligamentos', 'saldo']].sort_values('desligamentos', ascending=False).head(10)

Avaliar a distribuição dos indicadores

In [ ]:
indicadores_integrados[['2025', 'admissoes', 'saldo']].describe()

### Visualizando a distribuição 

In [ ]:
import numpy as np

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
# Plot de registros migratórios 

# Histograma com transformação logarítmica
np.log1p(indicadores_integrados["2025"].dropna()).plot(
    kind="hist",
    bins=30,
    figsize=(10, 5)
)

plt.title("Distribuição dos registros migratórios em 2025")
plt.xlabel("Log(1 + registros migratórios)")
plt.ylabel("Número de municípios")
plt.show()

In [ ]:
np.log1p(indicadores_integrados["admissoes"].dropna()).plot(
    kind="hist",
    bins=30,
    figsize=(10, 5)
)

plt.title("Distribuição dos admissoes em 2025")
plt.xlabel("Log(1 + admissoes)")
plt.ylabel("Número de municípios")
plt.show()

In [ ]:
# Investigando a relação entre migração e emprego 

indicadores_integrados.plot(
    kind="scatter",
    x='2025',
    y='admissoes',
)

plt.title("Relação entre registros migratórios e admissões")
plt.show()

In [ ]:
indicadores_integrados['log_migracao'] = np.log1p(indicadores_integrados['2025'])

indicadores_integrados['log_admissoes'] = np.log1p(indicadores_integrados['admissoes'])


indicadores_integrados.plot(
    kind="scatter",
    x='log_migracao',
    y='log_admissoes',
)

plt.title("Relação entre registros migratórios e admissões (log1p)")
plt.show()

### Calcular a correlação
Agora vamos quantificar essa associação utilizando o coeficiente de correlação de Pearson.

Ele varia de -1 a +1:
- Próximo de +1: associação linear positiva forte.
- Próximo de 0: pouca associação linear.
- Próximo de -1: associação linear negativa forte.
Como estamos investigando a relação observada no gráfico, vamos calcular a correlação entre as duas variáveis transformadas.

In [ ]:
correlacao = indicadores_integrados[
    ["log_migracao", "log_admissoes"]
].corr()

display(correlacao)

In [ ]:
correlacao_2 = indicadores_integrados[
    ["2025", "admissoes"]
].corr()

display(correlacao_2)

In [ ]:
correlacao_spearman = indicadores_integrados[
    ["2025", "admissoes"]
].corr(method='spearman')

display(correlacao_spearman)

In [ ]:
correlacao_saldo_spearman = indicadores_integrados[
    ["2025", "saldo"]
].corr(method='spearman')

display(correlacao_saldo_spearman)

In [ ]:
correlacao_admissoes_saldo_spearman = indicadores_integrados[
    ["admissoes", "saldo"]
].corr(method='spearman')

display(correlacao_admissoes_saldo_spearman)

In [ ]:
# Agrupa as movimentações por município e ocupação CBO
emprego_migrantes_cbo_2025 = (
    caged_atividades
    .groupby(["codigo_ibge", "cbo2002ocupacao"], dropna=False)
    .agg(
        admissoes=("saldomovimentacao", lambda x: (x == 1).sum()),
        desligamentos=("saldomovimentacao", lambda x: (x == -1).sum())
    )
    .reset_index()
)

# Calcula o saldo por município e ocupação
emprego_migrantes_cbo_2025["saldo"] = (
    emprego_migrantes_cbo_2025["admissoes"]
    - emprego_migrantes_cbo_2025["desligamentos"]
)

# Verifica as dimensões e os totais
print("Dimensões:", emprego_migrantes_cbo_2025.shape)
print("Admissões:", emprego_migrantes_cbo_2025["admissoes"].sum())
print("Desligamentos:", emprego_migrantes_cbo_2025["desligamentos"].sum())
print("Códigos IBGE ausentes:",
      emprego_migrantes_cbo_2025["codigo_ibge"].isna().sum())

In [ ]:
# Padroniza os códigos CBO como texto de seis dígitos
emprego_migrantes_cbo_2025["cbo2002ocupacao"] = (
    emprego_migrantes_cbo_2025["cbo2002ocupacao"]
    .astype("string")
    .str.zfill(6)
)

# Salva os indicadores de migrantes por município e ocupação
emprego_migrantes_cbo_2025.to_parquet(
    "../data/processed/trabalho/emprego_migrantes_municipio_cbo_2025.parquet",
    index=False
)

# Confirma a exportação
print("Arquivo salvo com sucesso!")